# 5.3 Social Factors

**Survey series, class 3.** Where people live, how much they earn, how much schooling they had, whether they have health insurance, and whether they can afford enough food all shape their health. These are the **social determinants of health**, and type 2 diabetes is one of the conditions most closely tied to them. In this class we code four social factors from the surveys, describe them in cases and controls, and use them to answer the question 5.2 left open: is the link between alcohol and diabetes explained by **confounding**?

### By the end of this notebook you'll be able to

1. Turn ordered answers (income, education) into sensible **groups**, and keep *Prefer not to answer* as a category
2. Find and code questions from the **Social Determinants of Health** survey
3. Compare social factors between cases and controls, and model them with **logistic regression**
4. Show **confounding** in practice: how an odds ratio changes when a confounder is added, on the **same people**
5. Explain why social factors are described as **associations**, and why adjusting for some variables can remove part of an effect

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Are income, education, health insurance, and food insecurity linked to type 2 diabetes? Does adjusting for them explain the alcohol result from 5.2? |
| 🧪 **The analysis** | Percents by group; **logistic regression** adjusted for age and sex; the alcohol model **with and without** income and education, on the same people |
| 🎯 **Why** | Social factors are strong, well-documented risk factors for type 2 diabetes, and common **confounders** of other links |
| ✅ **What we conclude** | How strongly each social factor is linked to type 2 diabetes, and how much of the alcohol link they explain |
| ⚠️ **What it can't tell us** | Cause. Diabetes can also **lower** income (costs, missed work); and these factors reflect wider circumstances (where people live, discrimination, access to care), not individual choices |
| 🔗 **Connections** | Uses files from **5.1** and **5.2**; income non-answers from **5.1**; the alcohol result from **5.2**; models from **3.2** |

| | |
|---|---|
| 📂 **Uses** | the *All of Us* Controlled Tier dataset; `cohort_sample.csv` (**1.1**); `survey_questions.csv` (**5.1**); `survey_lifestyle.csv` (**5.2**) |
| 💾 **Saves** | `survey_social.csv` (one row per person) |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup, and finding the questions | 10 |
| 3. Income, education, insurance | 20 |
| 4. Food insecurity | 10 |
| 5. Describe and model | 15 |
| 6. Confounding: alcohol, again | 15 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 90 minutes |
| ⏱️ **Run time** | about **3 minutes** to run every cell |
| 💰 **Data cost** | about **$1**: two queries on `ds_survey` |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$1.50** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Social determinants of health

The conditions people are born into, live, and work in: income, education, housing, food, transport, access to care, discrimination. They affect diabetes through many paths: what food people can afford, safe places to be active, stress, and whether diabetes is found and treated early.

| Factor | Survey | Question (short) |
|---|---|---|
| **Income** | The Basics | annual household income, in ranges |
| **Education** | The Basics | highest grade or degree completed |
| **Health insurance** | The Basics | do you have health insurance? |
| **Food insecurity** | Social Determinants of Health | worried that food would run out; food didn't last (the **Hunger Vital Sign**, two questions used in clinics) |

### Confounding, on the same people

In **5.2**, people who drank more often had **lower** odds of type 2 diabetes. One explanation: people with **higher incomes** and more **education** drink more often **and** have less diabetes. If so, income and education are **confounders**, and adding them to the model should move the alcohol odds ratios **toward 1**.

To see that change fairly, both models must use the **same people**. `glm()` drops anyone missing a variable, so a model with more variables can quietly use fewer people, and the odds ratios would differ for that reason alone.

### Confounder or step on the path?

Adjusting isn't always right. If low income leads to obesity, which leads to diabetes, then **BMI is on the path** from income to diabetes (a **mediator**). Adjusting for BMI would remove part of the income effect we're trying to measure. Before adding a variable, ask: does it come **before** the factor (a possible confounder), or **after** it (a possible mediator)?

### Careful words

These are **associations**. Social factors describe circumstances, not people's choices or biology; and diabetes can itself **lower** income (medical costs, missed work). Write "people with lower incomes had higher odds of type 2 diabetes", not "low income causes diabetes".

## Part 2. Setup

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_sample <- load_cohort_sample()
questions     <- load_data("survey_questions.csv")
lifestyle     <- load_data("survey_lifestyle.csv", col_types = cols(person_id = "c"))

ids_sql <- str_c(cohort_sample$person_id, collapse = ", ")   # don't print: participant IDs

answer_text <- function(answer) {   # from 5.1
  if_else(str_starts(answer, "PMI:"), answer, str_remove(answer, "^.*: "))
}

cohort_sample %>% count(cohort)

### The Basics questions, by concept code

In [ ]:
CODES <- c(income    = "Income_AnnualIncome",
           education = "EducationLevel_HighestGrade",
           insurance = "Insurance_HealthInsurance")

basics_questions <- questions %>%
  filter(concept_code %in% CODES) %>%
  mutate(name = names(CODES)[match(concept_code, CODES)])

basics_questions %>% select(name, survey, question_concept_id, concept_code, question, n_people)

**🔍 Read the output:** three rows, one per name. If one is missing, its code is spelled differently: search the dictionary, e.g. `questions %>% filter(str_detect(question, regex("insurance", ignore_case = TRUE)))`, and fix `CODES`.

### The food questions, by text

The Social Determinants of Health questions have short codes that are hard to guess, so we search their **text**:

In [ ]:
questions %>%
  filter(survey == "Social Determinants of Health",
         str_detect(question, regex("food", ignore_case = TRUE))) %>%
  select(question_concept_id, concept_code, question, n_people)

**🔍 Read the output:** the food questions in the survey. The two **Hunger Vital Sign** questions ask whether, in the past 12 months, people **worried** their food would run out before they had money to buy more, and whether the food they bought **didn't last**. We pick them by their text:

In [ ]:
food_questions <- questions %>%
  filter(survey == "Social Determinants of Health",
         str_detect(question, regex("worried.*food.*run out|food.*(didn.t|did not) last", ignore_case = TRUE))) %>%
  mutate(name = if_else(str_detect(question, regex("worried", ignore_case = TRUE)), "food_worry", "food_didnt_last"))

food_questions %>% select(name, question_concept_id, question, n_people)

QUESTIONS <- c(setNames(basics_questions$question_concept_id, basics_questions$name),
               setNames(food_questions$question_concept_id,  food_questions$name))
QUESTIONS

**🔍 Read the output:** two food questions (one each for `food_worry` and `food_didnt_last`), and `QUESTIONS` with five named IDs. If a food question is missing, look at its wording in the table above and adjust the pattern.

## Part 3. Income, education, insurance

💰 About $0.50: the five questions for our sample.

In [ ]:
answers <- run_query(str_glue("
  SELECT person_id, question_concept_id, answer, DATE(survey_datetime) AS survey_date
  FROM `{CURRENT_DATASET}.ds_survey`
  WHERE question_concept_id IN ({str_c(QUESTIONS, collapse = ', ')})
    AND person_id IN ({ids_sql})")) %>%
  mutate(person_id = as.character(person_id),
         name      = names(QUESTIONS)[match(question_concept_id, QUESTIONS)],
         answer    = answer_text(answer))

answers %>% count(name)

**🔍 Read the output:** close to 10,000 rows for each Basics question (everyone answers The Basics), and fewer for the food questions (only people who took the Social Determinants of Health survey).

### Income: from nine ranges to four groups

In 5.1 we saw nine income ranges, plus many *Prefer Not To Answer*. Nine groups, some small, give unstable odds ratios, so we **combine** neighbouring ranges. *Prefer not to answer* stays as its **own group**: 5.1 showed it's common, and more common in cases, so dropping it would change who is compared.

In [ ]:
answers %>% filter(name == "income") %>% count(answer) %>% hide_small_counts()

income <- answers %>%
  filter(name == "income") %>%
  transmute(person_id,
            income = case_when(
              answer %in% c("less 10k", "10k 25k")                   ~ "Under $25k",
              answer %in% c("25k 35k", "35k 50k")                    ~ "$25k to $50k",
              answer %in% c("50k 75k", "75k 100k")                   ~ "$50k to $100k",
              answer %in% c("100k 150k", "150k 200k", "more 200k")   ~ "$100k or more",
              answer == "PMI: Prefer Not To Answer"                  ~ "Prefer not to answer",
              TRUE                                                   ~ NA_character_),
            income = factor(income, levels = c("$100k or more", "$50k to $100k", "$25k to $50k",
                                               "Under $25k", "Prefer not to answer")))

income %>% count(income)

**🔍 Read the output:** the nine ranges, then the five groups. `NA` is *PMI: Skip* only. The **first** level, *$100k or more*, is the model's **reference**, so every odds ratio compares a group with the highest incomes.

### Education

In [ ]:
answers %>% filter(name == "education") %>% count(answer) %>% hide_small_counts()

**🔍 Read the output:** the answers, from *Never Attended* to *Advanced Degree*. We make four groups. Small groups (like *Never Attended*) go into *Less than high school*.

In [ ]:
education <- answers %>%
  filter(name == "education") %>%
  transmute(person_id,
            education = case_when(
              str_starts(answer, "PMI:")                                                      ~ NA_character_,
              str_detect(answer, regex("never|one through|five through|nine through", ignore_case = TRUE)) ~ "Less than high school",
              str_detect(answer, regex("twelve|GED", ignore_case = TRUE))                    ~ "High school or GED",
              str_detect(answer, regex("college one|one to three", ignore_case = TRUE))      ~ "Some college",
              str_detect(answer, regex("college graduate|advanced", ignore_case = TRUE))     ~ "College degree",
              TRUE                                                                           ~ "CHECK"),
            education = factor(education, levels = c("College degree", "Some college",
                                                     "High school or GED", "Less than high school", "CHECK")))

answers %>%
  filter(name == "education") %>%
  left_join(education, by = "person_id") %>%
  count(answer, education) %>%
  hide_small_counts()

**🔍 Read the output:** each answer and its group. Any answer in **CHECK** didn't match a rule: add it to the right group before going on. (We use `"CHECK"` rather than `NA` so that unmatched answers can't hide among the non-answers.)

### Health insurance

In [ ]:
answers %>% filter(name == "insurance") %>% count(answer) %>% hide_small_counts()

insurance <- answers %>%
  filter(name == "insurance") %>%
  transmute(person_id,
            insured = case_when(answer == "Yes" ~ "Yes",
                                answer == "No"  ~ "No",
                                TRUE            ~ NA_character_),
            insured = factor(insured, levels = c("Yes", "No")))

insurance %>% count(insured)

**🔍 Read the output:** most people have insurance. *Not sure* and the non-answers become `NA`.

## Part 4. Food insecurity

Both food questions have the same answers: *Often true*, *Sometimes true*, *Never true*. The usual screening rule: **food insecure** if *often* or *sometimes* true for **either** question.

In [ ]:
answers %>% filter(str_starts(name, "food")) %>% count(name, answer) %>% hide_small_counts()

food <- answers %>%
  filter(str_starts(name, "food")) %>%
  mutate(insecure = case_when(str_detect(answer, regex("often|sometimes", ignore_case = TRUE)) ~ TRUE,
                              str_detect(answer, regex("never", ignore_case = TRUE))           ~ FALSE,
                              TRUE                                                             ~ NA)) %>%
  group_by(person_id) %>%
  summarise(food_insecure = case_when(any(insecure, na.rm = TRUE) ~ TRUE,      # often/sometimes to either
                                      all(insecure %in% FALSE)    ~ FALSE,     # never true to both
                                      TRUE                        ~ NA))

food %>% count(food_insecure)

**🔍 Read the output:** the answers to each question, then one value per person who answered either. `NA` means non-answers only (or *never* to one and a skip to the other).

⚠️ **Denominator:** only people who took the Social Determinants of Health survey have a value: about half our sample, and fewer cases than controls (**5.1**). People who didn't take it are **unknown**, not "food secure".

## Part 5. Describe and model

In [ ]:
social <- cohort_sample %>%
  select(person_id, cohort, age, sex_at_birth) %>%
  left_join(income,    by = "person_id") %>%
  left_join(education, by = "person_id") %>%
  left_join(insurance, by = "person_id") %>%
  left_join(food,      by = "person_id") %>%
  mutate(t2d = as.integer(cohort == "case"), age10 = age / 10)

nrow(social)   # still 10,000
save_data(social %>% select(person_id, income, education, insured, food_insecure), "survey_social.csv")

describe <- function(data, var) {   # from 5.2
  data %>%
    filter(!is.na(.data[[var]])) %>%
    count(cohort, level = as.character(.data[[var]])) %>%
    hide_small_counts() %>%
    group_by(cohort) %>%
    mutate(percent = round(100 * n / sum(n, na.rm = TRUE), 1)) %>%
    ungroup() %>%
    select(-n) %>%
    pivot_wider(names_from = cohort, values_from = percent) %>%
    mutate(variable = var, .before = 1)
}

map(c("income", "education", "insured", "food_insecure"), ~ describe(social, .x)) %>%
  list_rbind() %>%
  print(n = 30)

**🔍 Read the output:** the percent of cases and controls in each group, among people who answered. Look for **gradients**: does the share of cases rise step by step as income or education falls?

### Logistic regression

Two models, because they use different people:

1. **Income, education, insurance**: The Basics, almost everyone
2. **Food insecurity**: only people who took Social Determinants of Health, adjusted for **income** too (food insecurity is partly about money: does it matter beyond income?)

In [ ]:
model_data <- social %>% filter(sex_at_birth %in% c("Female", "Male")) %>% droplevels()

m_social <- glm(t2d ~ age10 + sex_at_birth + income + education + insured,
                family = binomial, data = model_data)
m_food   <- glm(t2d ~ age10 + sex_at_birth + income + food_insecure,
                family = binomial, data = model_data)

c(social_model = nobs(m_social), food_model = nobs(m_food))

odds_ratios <- bind_rows(
  broom::tidy(m_social, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "income, education, insurance"),
  broom::tidy(m_food,   exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "food insecurity")) %>%
  filter(term != "(Intercept)") %>%
  transmute(model, term,
            OR      = round(estimate, 2),
            CI_low  = round(conf.low, 2),
            CI_high = round(conf.high, 2),
            p_value = format.pval(p.value, digits = 2, eps = 0.001))

odds_ratios %>% print(n = 30)

**🔍 Read the output:** odds ratios adjusted for age, sex, and the other factors in each model. References: *$100k or more*, *College degree*, insured, not food insecure.

- **Income** and **education**: look for odds ratios that **rise** step by step as income or education falls: a **gradient**.
- **Prefer not to answer** (income): its odds ratio shows whether the people who didn't answer look more like lower- or higher-income groups.
- **Food insecurity**: an odds ratio above 1 **after** adjusting for income means it matters beyond income alone.

Income and education are linked to each other, so each odds ratio is **adjusted** for the other: it shows what each adds beyond the other.

### A forest plot

In [ ]:
odds_ratios %>%
  filter(!term %in% c("age10", "sex_at_birthMale"), !(model == "food insecurity" & str_starts(term, "income"))) %>%
  mutate(group = case_when(str_starts(term, "income")    ~ "Income (vs $100k or more)",
                           str_starts(term, "education") ~ "Education (vs college degree)",
                           TRUE                          ~ "Insurance and food"),
         label = case_when(term == "insuredNo"         ~ "No health insurance",
                           term == "food_insecureTRUE" ~ "Food insecure",
                           TRUE ~ str_remove(term, "^income|^education")),
         label = fct_inorder(label)) %>%
  ggplot(aes(x = OR, y = fct_rev(label), colour = group)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbar(aes(xmin = CI_low, xmax = CI_high), width = 0.2) +
  geom_point(size = 3) +
  scale_x_log10() +
  labs(x = "Odds ratio (log scale)", y = NULL, colour = NULL,
       title = "Social factors and type 2 diabetes",
       subtitle = "Odds ratios adjusted for age and sex (and each other)") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom", legend.direction = "vertical")

### ✍️ Your turn

Health care access is another social factor. In the question dictionary, find a question from **Healthcare Access & Utilization** about **delaying** or **not getting** care because of **cost**. How many people answered it?

<details>
<summary><b>Show answer</b></summary>

```r
questions %>%
  filter(str_detect(survey, "Healthcare Access"),
         str_detect(question, regex("delay|afford|cost", ignore_case = TRUE))) %>%
  select(question_concept_id, concept_code, question, n_people) %>%
  arrange(desc(n_people))
```

Several questions ask about delaying care, or not getting care or medicines, because of cost. To use one, add its ID to `QUESTIONS`, rerun the query, and code it as in Part 4. Remember it's a later survey: about half our sample took it.

</details>

In [ ]:
# Your code here

## Part 6. Confounding: alcohol, again

In 5.2, drinking more often went with **lower** odds of type 2 diabetes (2+ times a week: OR about 0.26), and answers given before diagnosis didn't change that. Is it **confounding** by income and education?

We fit the 5.2 alcohol model **twice**, on the **same people**: first as in 5.2, then adding income and education. `drop_na()` keeps only people with every variable, so both models use exactly the same rows.

In [ ]:
confounding_data <- model_data %>%
  left_join(lifestyle %>% select(person_id, smoking_status, alcohol_use), by = "person_id") %>%
  mutate(smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current")),
         alcohol_use    = factor(alcohol_use, levels = c("Never drank", "None in past year", "Monthly or less",
                                                         "2-4 times a month", "2+ times a week"))) %>%
  drop_na(t2d, age10, sex_at_birth, smoking_status, alcohol_use, income, education)

nrow(confounding_data)

m_5_2      <- glm(t2d ~ age10 + sex_at_birth + smoking_status + alcohol_use,
                  family = binomial, data = confounding_data)
m_adjusted <- glm(t2d ~ age10 + sex_at_birth + smoking_status + alcohol_use + income + education,
                  family = binomial, data = confounding_data)

bind_rows(
  broom::tidy(m_5_2,      exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "as in 5.2"),
  broom::tidy(m_adjusted, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "+ income and education")) %>%
  filter(str_starts(term, "alcohol|smoking")) %>%
  transmute(term, model, OR = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high)) %>%
  pivot_wider(names_from = model, values_from = OR)

**🔍 Read the output:** the alcohol and smoking odds ratios, before and after adding income and education, for the same people.

- If the alcohol odds ratios move **toward 1**, income and education explain **part** of the link: they were confounders.
- How much is left? Any remaining gap can reflect other confounders we haven't measured (overall health, diet, physical activity, BMI), or something real. A survey analysis can't tell which.

**💬 Discuss:** should we also adjust for **BMI**? (BMI could be a confounder, if heavier people drink less; or on the **path**, if alcohol affects weight. The answer depends on the question you're asking, and on what comes first.)

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *'survey_lifestyle.csv' does not exist* | 5.2 hasn't been run | Run 5.2 |
| `basics_questions` has fewer than 3 rows | a concept code is spelled differently | search the dictionary by text, and fix `CODES` |
| `food_questions` has no rows, or more than two | the question wording differs from the pattern | look at the food question table, and adjust the pattern |
| An education answer is in **CHECK** | a rule doesn't match its text | add it to the right group |
| Two models use different numbers of people | different missing values | `drop_na()` on **all** the variables first (Part 6) |

## ✅ Key takeaways

- Combine ordered answers into **groups** with enough people; keep **Prefer not to answer** as its own group when it's common
- Find questions by **concept code** when you know it, by **text** when you don't, and always **check** the matches
- Look for **gradients** across income and education groups
- Show confounding by fitting models **with and without** a variable, on the **same people**
- Ask whether a variable is a **confounder** or on the **path** before adjusting for it
- Social factors are **associations**, reflecting circumstances, not choices or biology

**This is the end of the survey series.** The **wearables series** (6.x) adds Fitbit data: steps, sleep, and heart rate.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).

## 🧑‍🏫 Instructor notes

- **Language matters most in this class.** Model the wording: "people with lower incomes had higher odds", never "the poor get diabetes". Social factors reflect circumstances and structures, not individual choices.
- **Before Part 6,** ask students to predict: will the alcohol odds ratio move toward 1? By how much?
- **Reverse causation** applies here too: diabetes can reduce income and make it harder to work. Ask students for examples.
- **Likely question:** "Why not adjust for everything?" Adjusting for a variable on the path (a mediator) removes part of the effect you want to measure. Draw income → BMI → diabetes on the board.